Data Pipeline & ASX200 Universe Audit

This notebook handles the extraction, validation and unification of price history for the ASX 200 universe. 
It identifies the data gaps (from <3 years IPO or Ticker Changes) and bridges historical data from DatAnalysis (Morningstar) with live yfinance data 

In [ ]:
#IMPORTS & FOLDER SET UP  
import yfinance as yf
import pandas as pd
import os
import time
from datetime import datetime, timedelta

# Create directory structure
folders = ['data_downloads', 'manual_downloads']
for folder in folders:
    if not os.path.exists(folder):
        os.makedirs(folder)

# Target start date for a 3-year lookback
TARGET_START = datetime.now() - timedelta(days=3*365)


The list of ASX200 Companies were taken from MarketIndex.com (https://www.marketindex.com.au/asx200) and downloaded as a CSV. 

The names were scrapped and suspended companies were manually removed - resulting in this list of 198 stocks as of 05/05/2026

In [ ]:
#ASX200 NAME SCRAPPER 
df = pd.read_csv('asx200.csv')

# Pull first column, remove empty rows, and ensure only unique tickers
tickers = df.iloc[:, 0].dropna().unique().tolist()

clean_tickers = [str(t).strip().upper() for t in tickers if len(str(t).strip()) > 0]
clean_tickers.sort()

print(f"Total Unique Tickers found: {len(clean_tickers)}")
print("-" * 30)
print(clean_tickers)


In [ ]:
# FULL CONSTITUENT LIST (MAY 2026) - CTD and NSR suspended at 05/05/2026
asx_200_full = [
    '360', '4DX', 'A2M', 'AAI', 'AFI', 'AGL', 'AIA', 'ALD', 'ALK', 'ALL',
    'ALQ', 'ALX', 'AMC', 'AMP', 'ANN', 'ANZ', 'APA', 'APE', 'ARG', 'ASB',
    'ASK', 'ASX', 'AUB', 'AUI', 'AZJ', 'BEN', 'BFL', 'BGL', 'BHP', 'BOQ',
    'BPT', 'BRG', 'BSL', 'BWP', 'BXB', 'CAR', 'CBA', 'CBO', 'CDA', 'CEN',
    'CGF', 'CHC', 'CIA', 'CIP', 'CLW', 'CMM', 'CNU', 'COH', 'COL', 'CPU',
    'CQR', 'CSC', 'CSL', 'CWY', 'DBI', 'DNL', 'DOW', 'DRO', 'DRR', 'DVP', 
    'DXS', 'DYL', 'EBO', 'EDV', 'ELV', 'EMR', 'EOS', 'EVN', 'EVT', 'FBU',
    'FLT', 'FMG', 'FPH', 'FRW', 'GGP', 'GLF', 'GMD', 'GMG', 'GNE', 'GPT',
    'GQG', 'GYG', 'HDN', 'HUB', 'HVN', 'IAG', 'IFT', 'IGO', 'ILU', 'IMD',
    'JBH', 'JHX', 'L1G', 'LLC', 'LNW', 'LOV', 'LSF', 'LTR', 'LYC', 'MAQ',
    'MCY', 'MEZ', 'MFF', 'MFG', 'MGR', 'MIN', 'MND', 'MPL', 'MQG', 'MSB',
    'MTS', 'MXT', 'NAB', 'NEM', 'NHC', 'NHF', 'NIC', 'NST', 'NWH', 'NWL',
    'NXG', 'NXT', 'OBM', 'ORG', 'ORI', 'PDI', 'PDN', 'PLS', 'PME', 'PMV',
    'PNI', 'PPT', 'PRN', 'PRU', 'PXA', 'QAN', 'QBE', 'QUB', 'REA', 'REG',
    'REH', 'RGN', 'RHC', 'RIO', 'RMD', 'RMS', 'RRL', 'RSG', 'RWC', 'RYM',
    'S32', 'SCG', 'SDF', 'SEK', 'SFR', 'SGH', 'SGM', 'SGP', 'SHL', 'SIG',
    'SMR', 'SOL', 'SPK', 'SRG', 'SRL', 'STO', 'SUL', 'SUN', 'TAH', 'TCL',
    'TLC', 'TLS', 'TLX', 'TNE', 'TPG', 'TUA', 'TWE', 'VAU', 'VCX', 'VEA',
    'VGN', 'VNT', 'VUL', 'WAF', 'WAM', 'WBC', 'WDS', 'WES', 'WGX', 'WHC',
    'WLE', 'WOR', 'WOW', 'WTC', 'XRO', 'XYZ', 'YAL', 'ZIP'
]

In [ ]:
#AUDIT & DOWNLOAD FUNCTION
def final_asx200_audit(ticker_list, folder="data_downloads"):
    if not os.path.exists(folder): os.makedirs(folder)
    
    unfound = []
    success_count = 0
    target_start = datetime.now() - timedelta(days=3*365)
    
    print(f"--- Starting Final 200 Audit ---")

    for i, ticker in enumerate(ticker_list):
        if i % 15 == 0: time.sleep(1) #yfinance rate limit protection

        symbol = f"{ticker}.AX"
        try:
            df = yf.download(symbol, period="3y", auto_adjust=True, progress=False)
            
            if df.empty:
                print(f"❌ {ticker}: NO DATA FOUND")
                unfound.append({"ticker": ticker, "reason": "No data on Yahoo"})
                continue

            first_date = df.index.min().to_pydatetime().replace(tzinfo=None)
            
            # Gap detection for bridging
            if first_date <= target_start and len(df) >= 756: # 3 years of data (252 trading days/year)
                #Saving 'good' data to file 
                df[df.index >= target_start].to_csv(os.path.join(folder, f"{ticker}.csv"))
                success_count += 1
            else:
                reason = f"BRIDGE NEEDED: Starts {first_date.date()}"
                print(f"❌ {ticker}: {reason}")
                unfound.append({"ticker": ticker, "reason": reason})
                
        except Exception as e:
            unfound.append({"ticker": ticker, "reason": f"Error: {e}"})

    pd.DataFrame(unfound).to_csv("MANUAL_STOCK_DOWNLOAD_LIST.csv", index=False)
    print(f"\n--- AUDIT COMPLETE ---\n✅ Clean: {success_count}\n❌ Gaps/Bridge Needed: {len(unfound)}")

if __name__ == "__main__":
    final_asx200_audit(asx_200_full)


Bridging DatAnalysis Data and yfinance Data.

The following code merges the manually downloaded DatAnalysis CSVs (for ticker transitions, e.g. SQ2 -> XYZ) with the new live dataset. 
Ensuring a continuous 3-year history. 

In [ ]:
#MASTER MERGER TOOL
def master_merger(raw_folder='data_downloads', manual_folder='manual_downloads'):
    all_files = [f for f in os.listdir(raw_folder) if f.endswith('.csv')]
    master_list = []
    
    # Mapping for ticker transitions
    bridges = {'XYZ': 'SQ2.csv', 'NEM': 'NCM.csv', 'SGH': 'SVW.csv', 'DNL': 'IPL.csv'}

    for file in all_files:
        ticker = file.replace('.csv', '')
        df_new = pd.read_csv(os.path.join(raw_folder, file))
        df_new['ticker'] = ticker

        if ticker in bridges and os.path.exists(os.path.join(manual_folder, bridges[ticker])):
            df_old = pd.read_csv(os.path.join(manual_folder, bridges[ticker]))
            df_old['ticker'] = ticker
            
            # Align timezones and combine
            df_new['Date'] = pd.to_datetime(df_new['Date']).dt.tz_localize(None)
            df_old['Date'] = pd.to_datetime(df_old['Date']).dt.tz_localize(None)
            
            combined = pd.concat([df_old[df_old['Date'] < df_new['Date'].min()], df_new])
            master_list.append(combined.sort_values('Date'))
        else:
            master_list.append(df_new)

    final_df = pd.concat(master_list, ignore_index=True)
    final_df.to_csv("ASX200_MASTER_DATASET.csv", index=False)
    return final_df

master_data = master_merger()
print(f"Master dataset unified with {master_data['ticker'].nunique()} tickers.")